In [1]:
import xgboost as xgb


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = (
    Path.home() / "Downloads" / "IDX-Exchange-Data-Science"
)

# Load the saved feature-engineered datasets.
train_fe = pd.read_csv(
    PROJECT_DIR / "single_family_train_fe.csv",
    low_memory=False,
)

test_fe = pd.read_csv(
    PROJECT_DIR / "single_family_test_fe.csv",
    low_memory=False,
)

# Parse sale dates for chronological splitting only.
train_dates = pd.to_datetime(
    train_fe["CloseDate"], errors="raise", utc=True
)
test_dates = pd.to_datetime(
    test_fe["CloseDate"], errors="raise", utc=True
)

validation_start = pd.Timestamp("2026-03-01", tz="UTC")
test_start = pd.Timestamp("2026-04-01", tz="UTC")
test_end = pd.Timestamp("2026-05-01", tz="UTC")

assert train_dates.notna().all()
assert test_dates.notna().all()
assert train_dates.lt(test_start).all()
assert (
    test_dates.ge(test_start) & test_dates.lt(test_end)
).all()

# Reserve March for hyperparameter tuning.
fit_mask = train_dates.lt(validation_start)
val_mask = train_dates.ge(validation_start)

fit_df = train_fe.loc[fit_mask].copy()
val_df = train_fe.loc[val_mask].copy()

assert len(fit_df) > 0 and len(val_df) > 0

print("Fit records:", len(fit_df))
print("Validation records:", len(val_df))
print("Test records:", len(test_fe))

print("\nFit date range:")
print(train_dates.loc[fit_mask].min(), "to",
      train_dates.loc[fit_mask].max())

print("\nValidation date range:")
print(train_dates.loc[val_mask].min(), "to",
      train_dates.loc[val_mask].max())

Fit records: 267859
Validation records: 10851
Test records: 11684

Fit date range:
2024-01-01 00:00:00+00:00 to 2026-02-28 00:00:00+00:00

Validation date range:
2026-03-01 00:00:00+00:00 to 2026-03-31 00:00:00+00:00


In [3]:
from pathlib import Path

output_dir = (
    Path.home() / "Downloads" / "IDX-Exchange-Data-Science"
)

train_fe.to_csv(
    output_dir / "single_family_train_fe.csv",
    index=False,
)

test_fe.to_csv(
    output_dir / "single_family_test_fe.csv",
    index=False,
)



In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Remove the target, sale date, and district metadata.
excluded_columns = [
    "ClosePrice",
    "CloseDate",
    "SpatialHighDistrictCode",
    "SpatialHighDistrictType",
    "SpatialElementaryDistrictCode",
    "SpatialElementaryDistrictType",
]

X_fit = fit_df.drop(
    columns=excluded_columns, errors="ignore"
).copy()

X_val = val_df.drop(
    columns=excluded_columns, errors="ignore"
).copy()

assert X_fit.columns.equals(X_val.columns)

# Restore undefined bed/bath ratios.
for data in [X_fit, X_val]:
    data.loc[
        data["ZeroBathrooms"].eq(1), "BedBathRatio"
    ] = np.nan

# Identify feature types.
categorical_columns = X_fit.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

numeric_columns = [
    col for col in X_fit.columns
    if col not in categorical_columns
]

# Standardize categorical labels.
for data in [X_fit, X_val]:
    for col in categorical_columns:
        data[col] = (
            data[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace("", pd.NA)
            .fillna("unknown")
            .astype(str)
        )

    data[numeric_columns] = data[numeric_columns].replace(
        [np.inf, -np.inf], np.nan
    )

# Keep the same preprocessing approach as previous models.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
            categorical_columns,
        ),
        ("numeric", numeric_pipeline, numeric_columns),
    ],
    sparse_threshold=1.0,
)

# Learn preprocessing rules from Fit data only.
X_fit_ready = preprocessor.fit_transform(X_fit)
X_val_ready = preprocessor.transform(X_val)

# Prepare original-price and natural-log-price targets.
y_fit = pd.to_numeric(fit_df["ClosePrice"], errors="raise")
y_val = pd.to_numeric(val_df["ClosePrice"], errors="raise")

for target in [y_fit, y_val]:
    assert np.isfinite(target).all() and target.gt(0).all()

y_fit_log = np.log(y_fit)
y_val_log = np.log(y_val)

print("Fit matrix:", X_fit_ready.shape)
print("Validation matrix:", X_val_ready.shape)
print("Log-price targets ready.")

Fit matrix: (267859, 17512)
Validation matrix: (10851, 17512)
Log-price targets ready.


In [5]:
import time
import xgboost as xgb

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

# Initial configuration before hyperparameter tuning.
xgb_baseline = xgb.XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    n_estimators=500,
    max_depth=6,
    learning_rate=0.10,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=4,
)

start_time = time.perf_counter()

print("Training XGBoost...", flush=True)

# Train using natural-log sale prices.
xgb_baseline.fit(X_fit_ready, y_fit_log)

print(
    f"Training completed in "
    f"{time.perf_counter() - start_time:.1f} seconds."
)

# Evaluate on both log-price and original-price scales.
baseline_rows = []

for split, features, actual in [
    ("Fit", X_fit_ready, y_fit),
    ("Validation", X_val_ready, y_val),
]:
    log_predictions = xgb_baseline.predict(features)
    predictions = np.exp(log_predictions)

    baseline_rows.append({
        "Split": split,
        "R² (price)": r2_score(actual, predictions),
        "R² (log price)": r2_score(
            np.log(actual), log_predictions
        ),
        "MAE ($)": mean_absolute_error(actual, predictions),
        "RMSE ($)": np.sqrt(
            mean_squared_error(actual, predictions)
        ),
        "MAPE (%)": (
            mean_absolute_percentage_error(actual, predictions)
            * 100
        ),
    })

xgb_baseline_results = pd.DataFrame(baseline_rows)

display(
    xgb_baseline_results.style
    .hide(axis="index")
    .format({
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

Training XGBoost...
Training completed in 4.8 seconds.


Split,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
Fit,0.3808,0.9290,"157,469.62","1,194,150.13",12.01
Validation,0.8327,0.9222,"172,990.97","427,472.81",12.81


In [6]:
from sklearn.base import clone

tuning_rows = []
best_val_r2 = -np.inf
best_xgb = None
best_params = None

for depth in [4, 6, 8]:
    for lr in [0.05, 0.10]:
        print(
            f"Evaluating max_depth={depth}, learning_rate={lr}...",
            flush=True,
        )

        # Reuse the baseline when the parameters match.
        if depth == 6 and lr == 0.10:
            model = xgb_baseline
        else:
            model = clone(xgb_baseline).set_params(
                max_depth=depth,
                learning_rate=lr,
            )
            model.fit(X_fit_ready, y_fit_log)

        log_pred = model.predict(X_val_ready)
        price_pred = np.exp(log_pred)

        val_r2 = r2_score(y_val, price_pred)

        tuning_rows.append({
            "max_depth": depth,
            "learning_rate": lr,
            "n_estimators": 500,
            "R² (price)": val_r2,
            "R² (log price)": r2_score(y_val_log, log_pred),
            "MAE ($)": mean_absolute_error(y_val, price_pred),
            "RMSE ($)": np.sqrt(
                mean_squared_error(y_val, price_pred)
            ),
            "MAPE (%)": (
                mean_absolute_percentage_error(y_val, price_pred)
                * 100
            ),
        })

        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_xgb = model
            best_params = {
                "max_depth": depth,
                "learning_rate": lr,
                "n_estimators": 500,
            }

        print(f"Validation R² (price): {val_r2:.4f}")

tuning_results = pd.DataFrame(tuning_rows)

display(
    tuning_results
    .sort_values("R² (price)", ascending=False)
    .style.hide(axis="index")
    .format({
        "learning_rate": "{:.2f}",
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

print("Best parameters so far:", best_params)

Evaluating max_depth=4, learning_rate=0.05...
Validation R² (price): 0.7721
Evaluating max_depth=4, learning_rate=0.1...
Validation R² (price): 0.8015
Evaluating max_depth=6, learning_rate=0.05...
Validation R² (price): 0.8175
Evaluating max_depth=6, learning_rate=0.1...
Validation R² (price): 0.8327
Evaluating max_depth=8, learning_rate=0.05...
Validation R² (price): 0.8457
Evaluating max_depth=8, learning_rate=0.1...
Validation R² (price): 0.8624


max_depth,learning_rate,n_estimators,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
8,0.10,500,0.8624,0.9313,"159,363.25","387,659.98",11.88
8,0.05,500,0.8457,0.9242,"168,543.40","410,534.32",12.52
6,0.10,500,0.8327,0.9222,"172,990.97","427,472.81",12.81
6,0.05,500,0.8175,0.9105,"185,237.82","446,481.55",13.81
4,0.10,500,0.8015,0.9061,"192,804.26","465,669.19",14.31
4,0.05,500,0.7721,0.8854,"213,804.28","498,982.75",15.96


Best parameters so far: {'max_depth': 8, 'learning_rate': 0.1, 'n_estimators': 500}


### Light Hyperparameter Tuning

With `n_estimators` fixed at 500, we compared `max_depth` values of 4, 6, and 8 and `learning_rate` values of 0.05 and 0.10 using March 2026 validation data. The best combination was `max_depth=8` and `learning_rate=0.10`, achieving a validation R² (price) of 0.8624 and MAPE of 11.88%.

Next, we will hold these two parameters fixed and compare `n_estimators` values of 300, 500, and 800.

In [7]:
# Freeze the depth and learning rate selected in the previous stage.
selected_depth = best_params["max_depth"]
selected_lr = best_params["learning_rate"]

tree_count_rows = [
    row.copy()
    for row in tuning_rows
    if row["max_depth"] == selected_depth
    and row["learning_rate"] == selected_lr
]

for n_trees in [300, 800]:
    print(f"Training n_estimators={n_trees}...", flush=True)

    model = clone(xgb_baseline).set_params(
        max_depth=selected_depth,
        learning_rate=selected_lr,
        n_estimators=n_trees,
    )
    model.fit(X_fit_ready, y_fit_log)

    log_pred = model.predict(X_val_ready)
    price_pred = np.exp(log_pred)
    val_r2 = r2_score(y_val, price_pred)

    tree_count_rows.append({
        "max_depth": selected_depth,
        "learning_rate": selected_lr,
        "n_estimators": n_trees,
        "R² (price)": val_r2,
        "R² (log price)": r2_score(y_val_log, log_pred),
        "MAE ($)": mean_absolute_error(y_val, price_pred),
        "RMSE ($)": np.sqrt(
            mean_squared_error(y_val, price_pred)
        ),
        "MAPE (%)": (
            mean_absolute_percentage_error(y_val, price_pred)
            * 100
        ),
    })

    if val_r2 > best_val_r2:
        best_val_r2 = val_r2
        best_xgb = model
        best_params = {
            "max_depth": selected_depth,
            "learning_rate": selected_lr,
            "n_estimators": n_trees,
        }

    print(f"Validation R² (price): {val_r2:.4f}")

tree_count_results = pd.DataFrame(tree_count_rows)

display(
    tree_count_results
    .sort_values("R² (price)", ascending=False)
    .style.hide(axis="index")
    .format({
        "learning_rate": "{:.2f}",
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

print("Selected parameters:", best_params)

Training n_estimators=300...
Validation R² (price): 0.8526
Training n_estimators=800...
Validation R² (price): 0.8694


max_depth,learning_rate,n_estimators,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
8,0.10,800,0.8694,0.9347,"154,516.29","377,783.73",11.51
8,0.10,500,0.8624,0.9313,"159,363.25","387,659.98",11.88
8,0.10,300,0.8526,0.9266,"165,503.66","401,304.95",12.37


Selected parameters: {'max_depth': 8, 'learning_rate': 0.1, 'n_estimators': 800}


### Selection of n_estimators

With `max_depth=8` and `learning_rate=0.10` fixed, we compared 300, 500, and 800 trees. The 800-tree model achieved the best March validation performance, with R² (price) of 0.8694, RMSE of $377,783.73, and MAPE of 11.51%.

The selected parameters are `max_depth=8`, `learning_rate=0.10`, and `n_estimators=800`. Next, we will refit preprocessing and the model on all training records through March 2026, then evaluate on the April test set.

In [8]:
from sklearn.base import clone

# Prepare the full training set and April test set.
X_train_final = train_fe.drop(
    columns=excluded_columns, errors="ignore"
).copy()

X_test_final = test_fe.drop(
    columns=excluded_columns, errors="ignore"
).copy()

assert X_train_final.columns.equals(X_test_final.columns)
assert X_train_final.columns.equals(X_fit.columns)

# Apply the same feature cleaning used during tuning.
for data in [X_train_final, X_test_final]:
    data.loc[
        data["ZeroBathrooms"].eq(1), "BedBathRatio"
    ] = np.nan

    for col in categorical_columns:
        data[col] = (
            data[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace("", pd.NA)
            .fillna("unknown")
            .astype(str)
        )

    data[numeric_columns] = data[numeric_columns].replace(
        [np.inf, -np.inf], np.nan
    )

# Refit preprocessing on all training records through March.
preprocessor_final = clone(preprocessor)

X_train_final_ready = preprocessor_final.fit_transform(
    X_train_final
)
X_test_final_ready = preprocessor_final.transform(X_test_final)

y_train_final = pd.to_numeric(
    train_fe["ClosePrice"], errors="raise"
)
y_test_final = pd.to_numeric(
    test_fe["ClosePrice"], errors="raise"
)

for target in [y_train_final, y_test_final]:
    assert np.isfinite(target).all() and target.gt(0).all()

# Create a fresh model using the validation-selected parameters.
xgb_final = clone(xgb_baseline).set_params(**best_params)

print("Selected parameters:", best_params)
print("Training records:", len(y_train_final))
print("Test records:", len(y_test_final))

start_time = time.perf_counter()

print("Training final XGBoost...", flush=True)
xgb_final.fit(
    X_train_final_ready,
    np.log(y_train_final),
)

print(
    f"Training completed in "
    f"{time.perf_counter() - start_time:.1f} seconds."
)

# Evaluate the final model.
final_rows = []

for split, features, actual in [
    ("Training", X_train_final_ready, y_train_final),
    ("Test", X_test_final_ready, y_test_final),
]:
    log_predictions = xgb_final.predict(features)
    predictions = np.exp(log_predictions)

    if split == "Test":
        xgb_test_log_predictions = log_predictions
        xgb_test_predictions = predictions

    final_rows.append({
        "Model": "XGBoost",
        "Split": split,
        "R² (price)": r2_score(actual, predictions),
        "R² (log price)": r2_score(
            np.log(actual), log_predictions
        ),
        "MAE ($)": mean_absolute_error(actual, predictions),
        "RMSE ($)": np.sqrt(
            mean_squared_error(actual, predictions)
        ),
        "MAPE (%)": (
            mean_absolute_percentage_error(actual, predictions)
            * 100
        ),
    })

xgb_final_results = pd.DataFrame(final_rows)

display(
    xgb_final_results.style
    .hide(axis="index")
    .format({
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

Selected parameters: {'max_depth': 8, 'learning_rate': 0.1, 'n_estimators': 800}
Training records: 278710
Test records: 11684
Training final XGBoost...
Training completed in 14.8 seconds.


Model,Split,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
XGBoost,Training,0.4371,0.9518,"128,703.46","1,126,852.28",9.87
XGBoost,Test,0.8355,0.9343,"162,515.54","442,508.84",11.61


### XGBoost Results

After light hyperparameter tuning, we selected `max_depth=8`, `learning_rate=0.10`, and `n_estimators=800`. Preprocessing and the model were refitted on all 278,710 training records through March 2026.

On the April 2026 test set (11,684 records), XGBoost achieved R² (price) of 0.8355, R² (log price) of 0.9343.

In [9]:
%pip install lightgbm


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import lightgbm as lgb


In [12]:
import lightgbm as lgb

lgb_baseline = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=500,
    learning_rate=0.10,
    max_depth=6,
    num_leaves=31,
    subsample=0.9,
    subsample_freq=1,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=4,
    verbosity=-1,
)

start_time = time.perf_counter()

print("Training LightGBM...", flush=True)
lgb_baseline.fit(X_fit_ready, y_fit_log)

print(
    f"Training completed in "
    f"{time.perf_counter() - start_time:.1f} seconds."
)

lgb_baseline_rows = []

for split, features, actual in [
    ("Fit", X_fit_ready, y_fit),
    ("Validation", X_val_ready, y_val),
]:
    log_predictions = lgb_baseline.predict(features)
    predictions = np.exp(log_predictions)

    lgb_baseline_rows.append({
        "Model": "LightGBM",
        "Split": split,
        "R² (price)": r2_score(actual, predictions),
        "R² (log price)": r2_score(
            np.log(actual), log_predictions
        ),
        "MAE ($)": mean_absolute_error(actual, predictions),
        "RMSE ($)": np.sqrt(
            mean_squared_error(actual, predictions)
        ),
        "MAPE (%)": (
            mean_absolute_percentage_error(actual, predictions)
            * 100
        ),
    })

lgb_baseline_results = pd.DataFrame(lgb_baseline_rows)

display(
    lgb_baseline_results.style
    .hide(axis="index")
    .format({
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

Training LightGBM...
Training completed in 2.8 seconds.


Model,Split,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
LightGBM,Fit,0.3707,0.9236,"162,001.51","1,203,894.74",12.44
LightGBM,Validation,0.8359,0.9215,"172,953.67","423,410.97",12.89


In [13]:
from sklearn.base import clone

lgb_tuning_rows = []
lgb_best_val_r2 = -np.inf
lgb_best_model = None
lgb_best_params = None

for depth in [4, 6, 8]:
    for lr in [0.05, 0.10]:
        print(
            f"Evaluating max_depth={depth}, learning_rate={lr}...",
            flush=True,
        )

        # Reuse the existing baseline when parameters match.
        if depth == 6 and lr == 0.10:
            model = lgb_baseline
        else:
            model = clone(lgb_baseline).set_params(
                max_depth=depth,
                learning_rate=lr,
            )
            model.fit(X_fit_ready, y_fit_log)

        log_pred = model.predict(X_val_ready)
        price_pred = np.exp(log_pred)
        val_r2 = r2_score(y_val, price_pred)

        lgb_tuning_rows.append({
            "max_depth": depth,
            "num_leaves": 31,
            "learning_rate": lr,
            "n_estimators": 500,
            "R² (price)": val_r2,
            "R² (log price)": r2_score(y_val_log, log_pred),
            "MAE ($)": mean_absolute_error(y_val, price_pred),
            "RMSE ($)": np.sqrt(
                mean_squared_error(y_val, price_pred)
            ),
            "MAPE (%)": (
                mean_absolute_percentage_error(y_val, price_pred)
                * 100
            ),
        })

        if val_r2 > lgb_best_val_r2:
            lgb_best_val_r2 = val_r2
            lgb_best_model = model
            lgb_best_params = {
                "max_depth": depth,
                "learning_rate": lr,
                "n_estimators": 500,
                "num_leaves": 31,
            }

        print(f"Validation R² (price): {val_r2:.4f}")

lgb_tuning_results = pd.DataFrame(lgb_tuning_rows)

display(
    lgb_tuning_results
    .sort_values("R² (price)", ascending=False)
    .style.hide(axis="index")
    .format({
        "learning_rate": "{:.2f}",
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

print("Best LightGBM parameters so far:", lgb_best_params)

Evaluating max_depth=4, learning_rate=0.05...
Validation R² (price): 0.7737
Evaluating max_depth=4, learning_rate=0.1...
Validation R² (price): 0.8041
Evaluating max_depth=6, learning_rate=0.05...
Validation R² (price): 0.8063
Evaluating max_depth=6, learning_rate=0.1...
Validation R² (price): 0.8359
Evaluating max_depth=8, learning_rate=0.05...
Validation R² (price): 0.8179
Evaluating max_depth=8, learning_rate=0.1...
Validation R² (price): 0.8402


max_depth,num_leaves,learning_rate,n_estimators,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
8,31,0.10,500,0.8402,0.9268,"167,522.74","417,821.32",12.35
6,31,0.10,500,0.8359,0.9215,"172,953.67","423,410.97",12.89
8,31,0.05,500,0.8179,0.9156,"181,327.43","445,987.47",13.39
6,31,0.05,500,0.8063,0.9076,"190,428.71","459,991.04",14.15
4,31,0.10,500,0.8041,0.9062,"192,368.82","462,629.14",14.34
4,31,0.05,500,0.7737,0.8852,"214,419.45","497,239.13",15.98


Best LightGBM parameters so far: {'max_depth': 8, 'learning_rate': 0.1, 'n_estimators': 500, 'num_leaves': 31}


In [14]:
# Keep the selected depth, learning rate, and leaf limit fixed.
lgb_selected_depth = lgb_best_params["max_depth"]
lgb_selected_lr = lgb_best_params["learning_rate"]
lgb_selected_leaves = lgb_best_params["num_leaves"]

# Include the existing 500-tree result.
lgb_tree_rows = [
    row.copy()
    for row in lgb_tuning_rows
    if row["max_depth"] == lgb_selected_depth
    and row["learning_rate"] == lgb_selected_lr
]

for n_trees in [300, 800]:
    print(f"Training n_estimators={n_trees}...", flush=True)

    model = clone(lgb_baseline).set_params(
        max_depth=lgb_selected_depth,
        learning_rate=lgb_selected_lr,
        num_leaves=lgb_selected_leaves,
        n_estimators=n_trees,
    )
    model.fit(X_fit_ready, y_fit_log)

    log_pred = model.predict(X_val_ready)
    price_pred = np.exp(log_pred)
    val_r2 = r2_score(y_val, price_pred)

    lgb_tree_rows.append({
        "max_depth": lgb_selected_depth,
        "num_leaves": lgb_selected_leaves,
        "learning_rate": lgb_selected_lr,
        "n_estimators": n_trees,
        "R² (price)": val_r2,
        "R² (log price)": r2_score(y_val_log, log_pred),
        "MAE ($)": mean_absolute_error(y_val, price_pred),
        "RMSE ($)": np.sqrt(
            mean_squared_error(y_val, price_pred)
        ),
        "MAPE (%)": (
            mean_absolute_percentage_error(y_val, price_pred)
            * 100
        ),
    })

    if val_r2 > lgb_best_val_r2:
        lgb_best_val_r2 = val_r2
        lgb_best_model = model
        lgb_best_params = {
            "max_depth": lgb_selected_depth,
            "learning_rate": lgb_selected_lr,
            "num_leaves": lgb_selected_leaves,
            "n_estimators": n_trees,
        }

    print(f"Validation R² (price): {val_r2:.4f}")

lgb_tree_results = pd.DataFrame(lgb_tree_rows)

display(
    lgb_tree_results
    .sort_values("R² (price)", ascending=False)
    .style.hide(axis="index")
    .format({
        "learning_rate": "{:.2f}",
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

print("Selected LightGBM parameters:", lgb_best_params)

Training n_estimators=300...
Validation R² (price): 0.8249
Training n_estimators=800...
Validation R² (price): 0.8510


max_depth,num_leaves,learning_rate,n_estimators,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
8,31,0.10,800,0.8510,0.9325,"159,871.65","403,437.66",11.77
8,31,0.10,500,0.8402,0.9268,"167,522.74","417,821.32",12.35
8,31,0.10,300,0.8249,0.9185,"177,973.13","437,342.04",13.16


Selected LightGBM parameters: {'max_depth': 8, 'learning_rate': 0.1, 'num_leaves': 31, 'n_estimators': 800}


In [15]:
# Train a fresh LightGBM using the validation-selected parameters.
lgb_final = clone(lgb_baseline).set_params(**lgb_best_params)

start_time = time.perf_counter()

print("Training final LightGBM...", flush=True)
lgb_final.fit(
    X_train_final_ready,
    np.log(y_train_final),
)

print(
    f"Training completed in "
    f"{time.perf_counter() - start_time:.1f} seconds."
)

lgb_final_rows = []

for split, features, actual in [
    ("Training", X_train_final_ready, y_train_final),
    ("Test", X_test_final_ready, y_test_final),
]:
    log_predictions = lgb_final.predict(features)
    predictions = np.exp(log_predictions)

    if split == "Test":
        lgb_test_log_predictions = log_predictions
        lgb_test_predictions = predictions

    lgb_final_rows.append({
        "Model": "LightGBM",
        "Split": split,
        "R² (price)": r2_score(actual, predictions),
        "R² (log price)": r2_score(
            np.log(actual), log_predictions
        ),
        "MAE ($)": mean_absolute_error(actual, predictions),
        "RMSE ($)": np.sqrt(
            mean_squared_error(actual, predictions)
        ),
        "MAPE (%)": (
            mean_absolute_percentage_error(actual, predictions)
            * 100
        ),
    })

lgb_final_results = pd.DataFrame(lgb_final_rows)

display(
    lgb_final_results.style
    .hide(axis="index")
    .format({
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

Training final LightGBM...
Training completed in 4.9 seconds.


Model,Split,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
LightGBM,Training,0.3916,0.9366,"146,175.33","1,171,515.33",11.17
LightGBM,Test,0.8361,0.9332,"163,906.42","441,673.26",11.76


In [16]:
boosting_test_comparison = pd.concat(
    [
        xgb_final_results.query("Split == 'Test'"),
        lgb_final_results.query("Split == 'Test'"),
    ],
    ignore_index=True,
)

display(
    boosting_test_comparison.style
    .hide(axis="index")
    .format({
        "R² (price)": "{:.4f}",
        "R² (log price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

Model,Split,R² (price),R² (log price),MAE ($),RMSE ($),MAPE (%)
XGBoost,Test,0.8355,0.9343,"162,515.54","442,508.84",11.61
LightGBM,Test,0.8361,0.9332,"163,906.42","441,673.26",11.76


### Comparison of Tree-Based Models with Engineered Features

In [18]:
import json
from io import StringIO
from pathlib import Path

# Read saved feature-engineered tree results.
fe_path = Path.cwd() / "Feature Engineering.ipynb"

with fe_path.open(encoding="utf-8") as file:
    fe_notebook = json.load(file)

tree_table = None

for cell in fe_notebook["cells"]:
    source = "".join(cell.get("source", []))

    if "tree_results_fe = pd.DataFrame" not in source:
        continue

    for output in cell.get("outputs", []):
        html = output.get("data", {}).get("text/html", "")
        if isinstance(html, list):
            html = "".join(html)

        if "<table" in html:
            tree_table = pd.read_html(StringIO(html))[0]
            break

    if tree_table is not None:
        break

if tree_table is None:
    raise ValueError(
        "Tree results not found. Save the Feature Engineering "
        "notebook with its model result table first."
    )

columns = [
    "Model",
    "R² (price)",
    "MAE ($)",
    "RMSE ($)",
    "MAPE (%)",
]

# Combine April Test metrics for the four models.
tree_model_comparison = pd.concat(
    [
        tree_table.loc[tree_table["Split"].eq("Test"), columns],
        xgb_final_results.loc[
            xgb_final_results["Split"].eq("Test"), columns
        ],
        lgb_final_results.loc[
            lgb_final_results["Split"].eq("Test"), columns
        ],
    ],
    ignore_index=True,
)

# Convert formatted values from the saved table to numbers.
for col in columns[1:]:
    tree_model_comparison[col] = pd.to_numeric(
        tree_model_comparison[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip(),
        errors="raise",
    )

display(
    tree_model_comparison
    .sort_values("R² (price)", ascending=False)
    .style.hide(axis="index")
    .format({
        "R² (price)": "{:.4f}",
        "MAE ($)": "{:,.2f}",
        "RMSE ($)": "{:,.2f}",
        "MAPE (%)": "{:.2f}",
    })
)

Model,R² (price),MAE ($),RMSE ($),MAPE (%)
LightGBM,0.8361,"163,906.42","441,673.26",11.76
XGBoost,0.8355,"162,515.54","442,508.84",11.61
Random Forest,0.7707,"192,876.20","522,483.31",13.40
Decision Tree,0.7479,"204,703.81","547,840.88",14.38


### Conclusion

On the April 2026 test set, XGBoost and LightGBM outperformed Decision Tree and Random Forest across all reported metrics using engineered features. LightGBM achieved the highest R² (0.8361) and lowest RMSE (USD 441,673.26), while XGBoost achieved the lowest MAE (USD 162,515.54) and MAPE (11.61%). Overall, the two boosting models performed similarly.

In [19]:
log_columns = ["Model", "R² (log price)"]

log_model_comparison = pd.concat(
    [
        tree_table.loc[
            tree_table["Split"].eq("Test"), log_columns
        ],
        xgb_final_results.loc[
            xgb_final_results["Split"].eq("Test"), log_columns
        ],
        lgb_final_results.loc[
            lgb_final_results["Split"].eq("Test"), log_columns
        ],
    ],
    ignore_index=True,
)

log_model_comparison["R² (log price)"] = pd.to_numeric(
    log_model_comparison["R² (log price)"],
    errors="raise",
)

display(
    log_model_comparison
    .sort_values("R² (log price)", ascending=False)
    .style.hide(axis="index")
    .format({"R² (log price)": "{:.4f}"})
)

Model,R² (log price)
XGBoost,0.9343
LightGBM,0.9332
Random Forest,0.9094
Decision Tree,0.8953
